# **필요한 라이브러리 불러오기**

In [4]:
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
from torchvision.transforms.functional import to_pil_image
import torch.nn.functional as F

In [5]:
# GPU로 연산.
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 랜덤 시드 고정.
# 실험 조건을 동일하게 설정하여,
# 같은 input을 넣으면 같은 결과가 나올 수 있도록 함.
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

# **MNIST 데이터셋 불러오기**
- 60,000개의 train data, 10,000개의 test data

## ✏️ TODO: MNIST 학습·테스트 데이터 구분하기

PPT를 참고하여 `train` 인자의 빈칸을 채우세요.

- `mnist_train`: 모델 학습에 사용할 데이터이므로 `train=True`로 설정하세요.
- `mnist_test`: 모델 평가에 사용할 데이터이므로 `train=False`로 설정하세요.

In [7]:
# MNIST dataset
# 60,000개의 train data, 10,000개의 test data
mnist_train = torchvision.datasets.MNIST(root='MNIST_data/', # 다운로드 경로 지정
                                        train=True, # TODO: 학습 데이터 선택
                                        transform=transforms.ToTensor(), # 텐서로 변환
                                        download=True)

mnist_test = torchvision.datasets.MNIST(root='MNIST_data/', # 다운로드 경로 지정
                                       train=False, # TODO: 테스트 데이터 선택
                                       transform=transforms.ToTensor(), # 텐서로 변환
                                       download=True)

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.90MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 129kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.22MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.6MB/s]


In [8]:
# dataset 크기 출력
print(len(mnist_train))
print(len(mnist_test))

60000
10000


In [34]:
# 비교 실험에 사용할 batch size를 작성하세요.
batch_size = 128

train_dataset = mnist_train

# Data loader를 사용하여 배치 크기 지정
train_loader=torch.utils.data.DataLoader(train_dataset,
                                        batch_size=batch_size,
                                        shuffle=True)
test_loader = torch.utils.data.DataLoader(mnist_test,
                                          batch_size=batch_size,
                                          shuffle=False)

# **모델 설계**
- MLP with dropout

## ✏️ TODO: Dropout 정의하고 순전파에 적용하기

PPT를 참고하여 Dropout을 정의하고, `forward()`의 빈칸에 Dropout을 적용하는 코드를 작성하세요.

- `self.dropout`: `nn.Dropout()`을 사용하여 비활성화할 뉴런의 비율을 설정하세요.
- 값이 클수록 학습 중 더 많은 뉴런이 비활성화됩니다.
- `forward()`에서는 각 ReLU 활성화 함수 다음에 `self.dropout`을 적용하세요.
- Dropout을 적용한 결과를 다시 `x`에 저장하세요.
- 마지막 출력층에는 Dropout을 추가하지 마세요.

In [35]:
class Net(nn.Module):
    def __init__(self):
        super(Net,self).__init__()
        # input_layer = 28*28, hidden_layer1 = 512
        self.fc1 = nn.Linear(28*28, 1024)
        # hidden_layer1 = 512, hidden_layer2 = 512
        self.fc2 = nn.Linear(1024,1024)
        # hidden_layer2 = 512, output_layer = 10
        self.fc3 = nn.Linear(1024,10)

        # TODO: nn.Dropout()을 사용하여 Dropout 비율을 설정하세요.
        self.dropout = nn.Dropout(0.0)

    def forward(self,x):
        x = x.view(-1,28*28)  # flatten
        x = F.relu(self.fc1(x))

        # TODO: 첫 번째 은닉층의 출력에 Dropout을 적용하세요.
        x = self.dropout(x)

        x = F.relu(self.fc2(x))

        # TODO: 두 번째 은닉층의 출력에 Dropout을 적용하세요.
        x = self.dropout(x)

        x = self.fc3(x)

        return x

model = Net().to(device)   # GPU에서 연산 수행하기 위해 device로 보냄

# **Train: 모델 학습**

In [36]:
epochs = 30                 # 학습 횟수
learning_rate = 0.001          # 학습률

In [37]:
# cost function과 optimizer 정의
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

In [38]:
# 모델 학습
model.train()
loss_list = []

for epoch in range(epochs):
  avg_loss = 0.0
  total_loss = 0.0
  for iter, data in enumerate(train_loader):

    # gradient 초기화
    optimizer.zero_grad()

    images, labels = data
    images = images.to(device)
    labels = labels.to(device)

    # 모델에 이미지 입력
    output = model(images)
    loss = criterion(output,labels)

    # 파라미터 업데이트
    loss.backward()
    optimizer.step()

    # 손실값 기록
    avg_loss += loss.item()
    total_loss += loss.item()

    if (iter+1) % 100==0: #100으로 나누어지는 순번일 경우
      print('Train Epoch: {} [{}/{}]\tLoss: {:.5f}'.format(
      epoch, iter+1, len(train_loader), avg_loss/100.)) # 케이스 100개 마다 평균 loss 출력
      avg_loss = 0.0 # 누적 loss 값 초기화
  loss_list.append(total_loss/(iter+1))  # epoch마다 평균 loss를 loss_list에 추가
print('\nLearning finished!')

Train Epoch: 0 [100/469]	Loss: 0.48524
Train Epoch: 0 [200/469]	Loss: 0.20167
Train Epoch: 0 [300/469]	Loss: 0.14701
Train Epoch: 0 [400/469]	Loss: 0.13159
Train Epoch: 1 [100/469]	Loss: 0.08143
Train Epoch: 1 [200/469]	Loss: 0.08947
Train Epoch: 1 [300/469]	Loss: 0.08301
Train Epoch: 1 [400/469]	Loss: 0.07668
Train Epoch: 2 [100/469]	Loss: 0.04695
Train Epoch: 2 [200/469]	Loss: 0.05543
Train Epoch: 2 [300/469]	Loss: 0.05813
Train Epoch: 2 [400/469]	Loss: 0.05586
Train Epoch: 3 [100/469]	Loss: 0.03606
Train Epoch: 3 [200/469]	Loss: 0.03455
Train Epoch: 3 [300/469]	Loss: 0.03799
Train Epoch: 3 [400/469]	Loss: 0.02910
Train Epoch: 4 [100/469]	Loss: 0.02528
Train Epoch: 4 [200/469]	Loss: 0.02880
Train Epoch: 4 [300/469]	Loss: 0.03214
Train Epoch: 4 [400/469]	Loss: 0.03307
Train Epoch: 5 [100/469]	Loss: 0.01939
Train Epoch: 5 [200/469]	Loss: 0.02004
Train Epoch: 5 [300/469]	Loss: 0.02708
Train Epoch: 5 [400/469]	Loss: 0.02323
Train Epoch: 6 [100/469]	Loss: 0.01677
Train Epoch: 6 [200/469]	

# **Test: 모델 성능 확인 및 결과 저장**

사전 학습된 가중치를 불러오지 않고, 현재 실험에서 직접 학습한 모델을 평가합니다.

In [39]:
# 모델 성능 확인
with torch.no_grad(): # torch.no_grad(): gradient 계산을 수행하지 않음
  model.eval()
  accuracy = 0.0

  for iter, data in enumerate(test_loader):
    images, labels = data
    images = images.to(device)
    labels = labels.to(device)
    output = model(images)
    pred = torch.argmax(output,1) == labels
    accuracy += pred.float().sum()

  accuracy_value = (100 * accuracy / len(mnist_test)).item()

# 마지막 학습 loss와 테스트 Accuracy 출력
print('Loss: {:.4f}'.format(loss_list[-1]))
print('Accuracy: {:.2f}%'.format(accuracy_value))

Loss: 0.0107
Accuracy: 98.19%


## ✏️ 추가 실습: 파라미터를 조절하며 학습 결과 비교하기

Learning rate, Epoch, Dropout 비율을 변경하면서 모델의 학습 결과를 비교해 보세요.

- `learning_rate`: 가중치를 조정하는 정도
- `epochs`: 전체 학습 데이터를 반복하는 횟수
- Dropout 비율: 학습 중 무작위로 비활성화할 뉴런의 비율

한 번에 하나의 값만 변경하고, 나머지 조건은 동일하게 유지하세요.

- 매 실험마다 모델과 옵티마이저를 새로 생성하여 처음부터 학습하세요.
- 각 실험의 설정값, Loss와 Accuracy를 기록하여 비교하세요.
- Dropout 비율을 `0`으로 설정한 경우와 비교하여 적용 여부에 따른 차이를 확인하세요.